In [1]:
import json
from datetime import date
from typing import Dict
import math
import csv

STAT_NAMES = [
    "caps", "hold", "earlyhold", "latehold", "ndps", "returns",
    "quick_returns", "nrts", "pups", "handoffs", "prevent", "lateprevent"
]

with open("../data/bulkmaps.json", encoding="utf-8") as f:
    maps = json.load(f)

with open("smurfs.json") as f:
    smurfs = json.load(f)

def desmurf(player):
    player = player.replace("\\'", "'").replace("\\\\", "\\")
    if player in smurfs:
        return smurfs[player]
    return player

def row_to_dict(row):
    return {
        'match_id': int(row[0]),
        'map_name': maps[row[1]]['name'],
        'gamemode': maps[row[1]]['type'],
        'timestamp': int(row[2]),
        'duration': int(row[3]),
        'cap_diff': int(row[4]),
        'players': [
            {
                'name': desmurf(row[j + 5]),
                'stats': {
                    stat_name: int(row[j * len(STAT_NAMES) + 13 + i])
                    for i, stat_name in enumerate(STAT_NAMES)
                }
            }
            for j in range(8)
        ]
    }

with open("matchups_with_stats.csv") as f:
    reader = csv.reader(f)
    next(reader)
    matches = [row_to_dict(row) for row in reader]
    matches = [m for m in matches if m['gamemode'] == "nf"]

players = {}

In [2]:
NEW_PLAYER_ELO = -0.8
STARTING_VARIANCE = 0.6
MAX_VARIANCE_TO_COUNT_AS_KNOWN = 0.3
VARIANCE_TO_ADD_BACK = 0.0005
BASE_VARIANCE = 4
RED_ADVANTAGE = { 'default': 0.1 }
DIFF_MAPPING = [0, 0.4, 1.0, 1.9, 3.0, 4.2, -4.2, -3.0, -1.9, -1.0, -0.4]
STAT_SCORE_WEIGHT_BY_CAP_DIFF = 0.2
CAP_TO_WP_EXPONENT = 2.0
PLAYER_STAT_WEIGHTS = {
    "caps": 1.4,
    "hold": 2.1 / 3600,
    "ndps": -0.7,
    "returns": 0.8,
    "nrts": 0.0,
    "pups": 0.1,
    "handoffs": 0.8,
}
TEAM_STAT_WEIGHTS = {
    "caps": 0,
    "hold": 0.0 / 3600,
    "ndps": 0.0,
    "returns": 0.0,
    "nrts": 0.0,
    "pups": 0.1,
    "handoffs": -0.1,
}
RELATIVE_ELO_CORRECTION = 2.3

In [3]:
class Player:
    def __init__(self, name):
        self.name = name
        self.elo = NEW_PLAYER_ELO
        self.variance = STARTING_VARIANCE
        self.lowest_ever_variance = self.variance
        self.stat_farm_amount = 0
        self.high_certainty_play = 0

    def error_bar(self):
        return 1.96 * self.variance ** 0.5
    
    def update_rating(self, p, match, error, stat_score, team_avg_elo, total_variance):
        p['updates'] = {
            'elo_before': self.elo,
            'variance_before': self.variance,
        }
        share_of_variance = self.variance / total_variance

        # Calculate stat score (weighted based on newness)
        newness = self.variance / STARTING_VARIANCE
        stat_score = stat_score * newness

        # Apply a slight correction to stat score based on relative elo
        elo_vs_game_avg = self.elo - team_avg_elo
        stat_score -= elo_vs_game_avg * newness * RELATIVE_ELO_CORRECTION

        # Weight stat score higher in blowouts
        stat_score *= 1 + STAT_SCORE_WEIGHT_BY_CAP_DIFF * (abs(match['cap_diff']) - 1)

        # Apply Bayesian update (error times share of variance, plus stat score)
        update = error + stat_score
        self.elo += update * share_of_variance
        self.variance *= 1 - share_of_variance
        self.lowest_ever_variance = min(self.variance, self.lowest_ever_variance)

        # Track the update that was applied
        p['updates']['elo_update'] = self.elo - p['updates']['elo_before']

    def overperformance(self, decay=0.998):
        expected_wins, actual_wins = 0, 0
        expected_cap_diff, actual_cap_diff = 0, 0
        weighted_games_played = 0
        for m in matches:
            names = [p['name'] for p in m['players']]
            if self.name in names:
                is_red = self.name in names[:4]
                weight = decay ** (date.today() - date.fromtimestamp(m['timestamp'])).days
                weighted_games_played += weight
                expected_score = sum(players[p].elo for p in names[:4]) - sum(players[p].elo for p in names[4:]) +\
                    RED_ADVANTAGE[m['map_name'] if m['map_name'] in RED_ADVANTAGE else 'default']
                red_win_prob = 1 / (1 + CAP_TO_WP_EXPONENT ** -expected_score)
                if is_red:
                    expected_cap_diff += expected_score * weight
                    actual_cap_diff += m['cap_diff'] * weight
                    expected_wins += red_win_prob * weight
                    actual_wins += (1 if m['cap_diff'] > 0 else 0.5 if m['cap_diff'] == 0 else 0) * weight
                else:
                    expected_cap_diff += expected_score * weight
                    actual_cap_diff += m['cap_diff'] * weight
                    expected_wins += (1 - red_win_prob) * weight
                    actual_wins += (1 if m['cap_diff'] < 0 else 0.5 if m['cap_diff'] == 0 else 0) * weight

        return (
            (actual_wins - expected_wins) / weighted_games_played,
            (actual_cap_diff - expected_cap_diff) / weighted_games_played
        )

    def __str__(self):
        return f"{self.name} ({self.elo:.2f} ± {self.error_bar():.2f})"
    
    def __repr__(self):
        return f"Player \"{self.name}\" ({self.elo:.2f} ± {self.error_bar():.2f})"

In [4]:
def normalize_elos():
    known_elos = [p.elo for p in players.values() if p.variance <= MAX_VARIANCE_TO_COUNT_AS_KNOWN]
    avg_elo = sum(known_elos) / max(10, len(known_elos))
    for p in players.values():
        p.elo -= avg_elo
        p.variance = min(p.variance + VARIANCE_TO_ADD_BACK, STARTING_VARIANCE)

def stat_scores(players, scoring_dict: Dict[str, int]):
    return [
        sum([
            p['stats'][stat_name] * scoring_dict[stat_name]
            for stat_name in scoring_dict
        ])
        for p in players
    ]

def judge_player_stats(players):
    player_scores = stat_scores(players, PLAYER_STAT_WEIGHTS)

    red_avg_score = sum(player_scores[:4]) / 4
    blue_avg_score = sum(player_scores[4:]) / 4
    red_scores = [s - red_avg_score for s in player_scores[:4]]
    blue_scores = [s - blue_avg_score for s in player_scores[4:]]
    return red_scores, blue_scores

def update_ratings(match):
    # Calculate skill for each team and game variance
    red_elo = 0
    blue_elo = 0
    total_variance = BASE_VARIANCE

    for p in match['players']:
        if p['name'] not in players:
            players[p['name']] = Player(p['name'])
        total_variance += players[p['name']].variance
    
    for p in match['players'][:4]:
        red_elo += players[p['name']].elo
    for p in match['players'][4:]:
        blue_elo += players[p['name']].elo
    
    expected_score = red_elo - blue_elo + RED_ADVANTAGE[
        match['map_name'] if match['map_name'] in RED_ADVANTAGE else 'default'
    ]

    # Track predictions
    match['red_elo'] = red_elo
    match['blue_elo'] = blue_elo
    match['pred'] = expected_score
    match['sd'] = total_variance ** 0.5
    match['wp'] = 1 / (1 + CAP_TO_WP_EXPONENT ** -expected_score)
    match['residual'] = match['cap_diff'] - min(max(expected_score, -5), 5)
    match['wwp'] = match['wp'] if match['cap_diff'] > 0 else 1 - match['wp']
    
    # Calculate the error used for Bayesian updating
    error = DIFF_MAPPING[match['cap_diff']] - min(max(expected_score, DIFF_MAPPING[-4]), DIFF_MAPPING[4])

    # Incorporate team total stats into the error
    team_scores = stat_scores(match['players'], TEAM_STAT_WEIGHTS)
    error += sum(team_scores[:4]) - sum(team_scores[4:])

    # Update ratings based on this error and the variance
    red_stat_scores, blue_stat_scores = judge_player_stats(match['players'])
    for p, stat_score in zip(match['players'][:4], red_stat_scores):
        players[p['name']].update_rating(p, match, error, stat_score, red_elo / 4, total_variance)
    for p, stat_score in zip(match['players'][4:], blue_stat_scores):
        players[p['name']].update_rating(p, match, -error, stat_score, blue_elo / 4, total_variance)

In [7]:
players = {}
day = date.fromtimestamp(matches[0]['timestamp'])
player_to_track = ""
played_today = 0

for match in matches:
    if any(p['name'] == player_to_track for p in match['players']):
        played_today += 1
    new_day = date.fromtimestamp(match['timestamp'])
    if new_day != day:
        normalize_elos()
        if played_today > 0:
            print(f"{new_day}: {players[player_to_track].elo:.2f} ± {players[player_to_track].error_bar():.2f} ({played_today} games)")
            played_today = 0
    day = new_day
    update_ratings(match)

print(f"MAE: {sum([abs(m['residual']) for m in matches]) / len(matches):.4f}    RMS: {(sum([m['residual'] ** 2 for m in matches]) / len(matches)) ** 0.5:.4f}")
print(f"COR: {sum([(1 if m['wwp'] > 0.5 else 0.5 if m['wwp'] == 0.5 else 0) for m in matches]) / len(matches):.2%}    WWP: {sum(m['wwp'] for m in matches) / len(matches):.2%}")
print(f"BRI: {1 - (sum([(1 - m['wwp']) ** 2 for m in matches]) / len(matches)) ** 0.5:.2%}    LOG: {2 ** (sum([math.log2(m['wwp']) for m in matches]) / len(matches)):.2%}")

for p in players.values():
    p.elo *= 0.8
    p.variance *= 1.25 ** 2

day = date.fromtimestamp(matches[0]['timestamp'])

for match in matches:
    new_day = date.fromtimestamp(match['timestamp'])
    if new_day != day:
        normalize_elos()
    day = new_day
    update_ratings(match)

print(f"MAE: {sum([abs(m['residual']) for m in matches]) / len(matches):.4f}    RMS: {(sum([m['residual'] ** 2 for m in matches]) / len(matches)) ** 0.5:.4f}")
print(f"COR: {sum([(1 if m['wwp'] > 0.5 else 0.5 if m['wwp'] == 0.5 else 0) for m in matches]) / len(matches):.2%}    WWP: {sum(m['wwp'] for m in matches) / len(matches):.2%}")
print(f"BRI: {1 - (sum([(1 - m['wwp']) ** 2 for m in matches]) / len(matches)) ** 0.5:.2%}    LOG: {2 ** (sum([math.log2(m['wwp']) for m in matches]) / len(matches)):.2%}")

MAE: 2.1970    RMS: 2.7063
COR: 67.32%    WWP: 58.83%
BRI: 54.51%    LOG: 54.84%
MAE: 2.0880    RMS: 2.5648
COR: 70.54%    WWP: 59.76%
BRI: 56.00%    LOG: 56.62%


In [8]:
leaderboard = sorted([
    p for p in players.values() if p.variance <= 0.16
], key=lambda p: p.elo, reverse=True)
for i, p in enumerate(leaderboard):
    print(f"{i + 1:>3}.  {p.name:<12}  {p.elo:5.2f} ± {1.96 * p.variance ** 0.5:.2f}")

  1.  Mileena        2.57 ± 0.31
  2.  tng.           2.34 ± 0.42
  3.  Suchit         2.14 ± 0.24
  4.  chizu_mizu     2.06 ± 0.30
  5.  meowza         2.03 ± 0.41
  6.  jig            2.02 ± 0.31
  7.  ?              2.00 ± 0.38
  8.  Vorhees        1.91 ± 0.26
  9.  OuchMyBalls    1.90 ± 0.36
 10.  Ball-E         1.82 ± 0.44
 11.  name333        1.80 ± 0.24
 12.  Cape           1.73 ± 0.66
 13.  womp womp      1.72 ± 0.77
 14.  black orchid   1.65 ± 0.32
 15.  2Nutz          1.63 ± 0.43
 16.  fender         1.58 ± 0.59
 17.  popsic         1.55 ± 0.64
 18.  TagTitan       1.52 ± 0.36
 19.  MarcusYallow   1.50 ± 0.58
 20.  phreak         1.50 ± 0.47
 21.  simp.          1.49 ± 0.72
 22.  autumn heart   1.45 ± 0.44
 23.  TheG           1.41 ± 0.42
 24.  DiNgBaT        1.40 ± 0.29
 25.  yep            1.39 ± 0.59
 26.  yawn           1.36 ± 0.37
 27.  tha king       1.33 ± 0.29
 28.  Vader          1.33 ± 0.34
 29.  Mr awesome:)   1.32 ± 0.38
 30.  churt chimi    1.32 ± 0.46
 31.  real

In [11]:
for m in matches:
    for p in m['players']:
        p['updates']['final_elo'] = players[p['name']].elo
        p['updates']['final_var'] = players[p['name']].variance
    red_elo = sum(players[p['name']].elo for p in m['players'][:4])
    blue_elo = sum(players[p['name']].elo for p in m['players'][4:])
    retro_pred = red_elo - blue_elo + RED_ADVANTAGE[
        match['map_name'] if match['map_name'] in RED_ADVANTAGE else 'default'
    ]
    retro_var = BASE_VARIANCE + sum(players[p['name']].variance for p in m['players'])
    retro_wp = 1 / (1 + CAP_TO_WP_EXPONENT ** -retro_pred)
    m['retro_red_elo'] = red_elo
    m['retro_blue_elo'] = blue_elo
    m['retro_pred'] = retro_pred
    m['retro_residual'] = m['cap_diff'] - m['retro_pred']
    m['retro_sd'] = retro_var ** 0.5
    m['retro_wp'] = retro_wp
    m['retro_wwp'] = retro_wp if m['cap_diff'] > 0 else 1 - retro_wp

print(f"MAE: {sum([abs(m['retro_residual']) for m in matches]) / len(matches):.4f}    RMS: {(sum([m['retro_residual'] ** 2 for m in matches]) / len(matches)) ** 0.5:.4f}")
print(f"COR: {sum([(1 if m['retro_wwp'] > 0.5 else 0.5 if m['retro_wwp'] == 0.5 else 0) for m in matches]) / len(matches):.2%}    WWP: {sum(m['retro_wwp'] for m in matches) / len(matches):.2%}")
print(f"BRI: {1 - (sum([(1 - m['retro_wwp']) ** 2 for m in matches]) / len(matches)) ** 0.5:.2%}    LOG: {2 ** (sum([math.log2(m['retro_wwp']) for m in matches]) / len(matches)):.2%}")

MAE: 1.9953    RMS: 2.4451
COR: 73.45%    WWP: 61.38%
BRI: 57.25%    LOG: 58.08%


### Records

In [13]:
def pretty_print_match(m, player=None):
    player_names = sorted([p['name'] for p in m['players'][:4]], key=lambda p: players[p].elo, reverse=True) + sorted([p['name'] for p in m['players'][4:]], key=lambda p: players[p].elo, reverse=True)
    player_caps = [p['stats']['caps'] for p in m['players']]
    minutes = int(m['duration'] / 3600)
    seconds = int(m['duration'] / 60) - 60 * minutes
    result = f"""Match {m['match_id']}: {', '.join(player_names[:4])} vs. {', '.join(player_names[4:])}
Expected: {m['retro_pred']:+.1f} ({m['retro_wp']:.0%}). Actual: {sum(player_caps[:4])}-{sum(player_caps[4:])} in {minutes}:{seconds:02}."""
    for p in m['players']:
        if p['name'] == player:
            result += f" Update: {p['updates']['elo_update']:+.2f}."
    return result + f" Map: {m['map_name']}\n"

def unfairness(m):
    return abs(m['retro_pred'])

def carry_required(m):
    return max(
        m['retro_blue_elo'] - m['retro_red_elo'] + max(players[p['name']].elo for p in m['players'][:4]),
        m['retro_red_elo'] - m['retro_blue_elo'] + max(players[p['name']].elo for p in m['players'][4:]),
    )

def matches_with_player(name):
    return [m for m in matches if name in [p['name'] for p in m['players']]]

# Biggest carries
for m in sorted([m for m in matches if m['retro_wwp'] < 0.5], key=unfairness, reverse=True)[:10]:
    print(pretty_print_match(m))

Match 4138291: black orchid, Irony, jjpoole, Oddball vs. Suchit, Vorhees, Ball-E, q42
Expected: -3.9 (6%). Actual: 6-5 in 8:09. Map: Arti NS

Match 4137163: Mileena, DiNgBaT, PTF, toppy vs. yep, Doris, Dante Ball, whiff
Expected: +3.6 (92%). Actual: 1-2 in 8:11. Map: Lockhart [MM23 South Winner]

Match 4140921: Magnolia, caprae, d0pe, 13.5 inches vs. TheG, DiNgBaT, Irony, trippi
Expected: -3.5 (8%). Actual: 6-5 in 8:28. Map: Carrera NFC

Match 4140220: tnevlos, PenFifteen, Daniel, Fatback slim vs. Carrrrrl, Penis Wings, cereballsy, helprefugees
Expected: -3.3 (9%). Actual: 5-4 in 8:23. Map: Wamble NFC

Match 4139400: Suchit, Jinjo, Carrrrrl, tbiol vs. Irony, Toidi, sholmes, BaII
Expected: +3.2 (90%). Actual: 5-6 in 9:45. Map: Arti NS

Match 4155223: Mileena, fender, known, Sadness vs. Vorhees, blah-zay, omni, phlasid
Expected: +3.1 (89%). Actual: 7-8 in 8:59. Map: wildflower

Match 4140873: meowza, Vader, RedBull, Marijuana vs. BallsToYou., trippi, clamp, S. Strasball
Expected: +2.9 (8

In [14]:
player_recent_matches = matches_with_player("Tumblewood")
player_recent_matches.reverse()
for m in player_recent_matches[:10]:
    print(pretty_print_match(m, player="Tumblewood"))

Match 4162504: smoji_, jerm, PenFifteen, Sussy Blocka vs. Tumblewood, Airmigo, timeboy, glizzy23
Expected: +0.7 (62%). Actual: 6-1 in 6:26. Update: -0.43. Map: Wamble NFC

Match 4161389: AdmaniaYT, ocd, TeaForYou&Me, glizzy23 vs. jkxs, Tumblewood, t h C, SumtimesGood
Expected: -2.4 (16%). Actual: 0-5 in 6:16. Update: +0.12. Map: Wombo Combo 2026

Match 4158571: jjpoole, rolf, Korgmonkey, PrettyLights vs. Dr. Toboggan, Tumblewood, T.Bomballdil, Liddi
Expected: +0.4 (56%). Actual: 7-3 in 8:00. Update: +0.09. Map: Wombo Combo 2026

Match 4158564: jjpoole, Korgmonkey, Ragnar, B) vs. Tumblewood, rolf, PrettyLights, T.Bomballdil
Expected: -0.5 (42%). Actual: 4-9 in 6:49. Update: +0.31. Map: wildflower

Match 4157267: Suchit, Rabz, SaladRT, Jarek vs. Tumblewood, broseph, sieh, smalls
Expected: -1.9 (21%). Actual: 4-5 in 8:13. Update: +0.18. Map: Wamble NFC



### Analysis

In [ ]:
for i in range(0, 15):
    i *= 0.4
    total = [m for m in matches if i < abs(m['retro_pred']) < i + 0.4]
    wins = [m for m in total if m['retro_wwp'] > 0.5]
    print(f"{i:.1f}-{i + 0.4:.1f}: "
          f"{len(wins):>5} / "
          f"{len(total):>5} "
          f"({len(wins) / len(total):.1%})"
          f"  {1 / (1 + 2.3 ** -(i + 0.2)):.1%}")

0.0-0.4:   198 /   392 (50.5%)  54.2%
0.4-0.8:   243 /   375 (64.8%)  62.2%
0.8-1.2:   220 /   303 (72.6%)  69.7%
1.2-1.6:   205 /   257 (79.8%)  76.2%
1.6-2.0:   176 /   212 (83.0%)  81.7%
2.0-2.4:   120 /   148 (81.1%)  86.2%
2.4-2.8:   106 /   116 (91.4%)  89.7%
2.8-3.2:    92 /   101 (91.1%)  92.4%
3.2-3.6:    59 /    65 (90.8%)  94.4%
3.6-4.0:    38 /    38 (100.0%)  95.9%
4.0-4.4:    27 /    28 (96.4%)  97.1%
4.4-4.8:    20 /    21 (95.2%)  97.9%
4.8-5.2:    13 /    13 (100.0%)  98.5%
5.2-5.6:    12 /    12 (100.0%)  98.9%
5.6-6.0:     4 /     4 (100.0%)  99.2%


In [ ]:
resids = {}
for m in matches:
    for i, p in enumerate(m['players']):
        if p['name'] not in resids:
            resids[p['name']] = [0, 0, 0]
        resids[p['name']][0] += 1
        resids[p['name']][1] += m['retro_residual'] * (1 if i < 4 else -1)
        resids[p['name']][2] += (1 - m['retro_wp']) if ((m['cap_diff'] > 0) == (i < 4)) else m['retro_wp']

for p in resids:
    resids[p][1] /= resids[p][0]
    resids[p][2] /= resids[p][0]

for m, p in sorted(resids.items(), key=lambda x: x[1][0], reverse=True)[:500]:
    if abs(p[1]) > 0.25:
        print(f"{m:>12}  {p[0]:>4}  {p[1]:+.2f}  {p[2]:.1%}")

         AJ.   135  -0.46  49.2%
    Sambal 1    94  +0.46  48.3%
       known    78  -0.29  46.3%
     Mileena    69  -0.51  56.3%
        tng.    65  +0.32  47.9%
       clamp    60  +0.31  55.2%
         Ivy    60  -0.35  45.2%
JukeBerallta    58  -0.42  53.7%
         K O    58  +0.63  52.7%
helprefugees    57  +0.28  53.9%
    MrDMoney    56  -0.45  50.9%
       Messy    56  +0.29  50.4%
  HammerTime    55  -0.40  47.5%
   `ArryKane    54  +0.51  51.0%
   BallSaget    53  -0.54  56.8%
      Borgus    53  -0.27  53.6%
          mc    53  -0.38  49.0%
  Gvendolino    53  +0.63  46.5%
SteroidSurge    53  +0.28  48.6%
Barbara Bush    52  +0.44  55.3%
      N()()B    50  +0.26  51.9%
        deku    47  -0.33  46.3%
      snakes    46  +0.26  49.4%
       ipopu    45  +0.26  49.6%
     Atticus    41  +0.48  49.3%
        Rayn    40  +0.45  55.7%
    ROBOPOP_    40  -0.34  47.3%
 GetPupsDeep    40  +0.75  58.4%
Mista Feeneh    39  -0.34  49.4%
       Big D    37  -0.68  49.1%
      cowb

In [ ]:
maps = {}
for m in matches:
    if m['map_name'] not in maps:
        maps[m['map_name']] = [0, 0, 0, 0, 0, 0]
    maps[m['map_name']][0] += 1
    maps[m['map_name']][3] += m['retro_residual']
    maps[m['map_name']][1] += math.log2(0.5 + abs(m['retro_wp'] - 0.5))
    maps[m['map_name']][2] += math.log2(m['retro_wwp'])
    maps[m['map_name']][4] += abs(m['retro_residual'])
    maps[m['map_name']][5] += m['retro_residual'] ** 2
for m in maps.values():
    if m[0] == 0:
        continue
    m[1] /= m[0]
    m[2] /= m[0]
    m[3] /= m[0]
    m[4] /= m[0]
    m[5] /= m[0]

for m, p in sorted(maps.items(), key=lambda x: x[1][0], reverse=True):
    print(f"{m:>28}  {p[0]:>4}  {2 ** p[1]:.1%}  {2 ** p[2]:.1%}  {p[3] + RED_ADVANTAGE[m if m in RED_ADVANTAGE else 'default']:+.2f}  {p[4]:.2f}  {p[5]:.2f}")

                   Gumbo NFC   227  69.6%  59.7%  +0.10  1.81  5.08
                Whiplash NFC   214  70.7%  61.2%  +0.22  1.98  5.70
                     Arti NS   202  68.3%  59.5%  +0.06  1.95  5.88
                 Almond 2023   199  70.0%  61.7%  -0.06  1.84  5.34
                  Wamble NFC   184  71.0%  61.0%  +0.07  1.79  4.99
Lockhart [MM23 South Winner]   172  68.2%  58.0%  -0.25  1.79  5.39
                Bulldog 2023   141  70.4%  59.3%  -0.45  2.00  6.36
                     REDLINE   119  68.6%  60.2%  +0.24  1.73  4.46
                 Plasma 2026   109  67.5%  57.9%  +0.16  1.91  5.50
                  wildflower   108  67.2%  56.5%  -0.40  2.95  11.42
                 Carrera NFC   107  69.3%  55.6%  +0.22  2.10  6.51
            Wombo Combo 2026    86  68.8%  56.5%  +0.32  1.88  5.76
                  Plasma NFC    71  71.1%  58.8%  -0.05  1.81  5.33
             Wombo Combo NFC    70  71.3%  54.7%  -0.46  1.89  5.13
             Sky Dweller NFC    55  66.3%  59.0

### Results

Guessing 0:

```
MAE: 2.6620    RMS: 3.1089
COR: 50.00%    WWP: 50.00%
BRI: 50.00%    LOG: 50.00%
```

Current system:

```
MAE: 2.1410    RMS: 2.6591
COR: 71.06%    WWP: 60.95%
BRI: 55.89%    LOG: 56.40%
```

With hindsight:

```
MAE: 1.8874    RMS: 2.3432
COR: 75.18%    WWP: 64.23%
BRI: 58.70%    LOG: 59.89%
```